# ANP-DJTLED on a Colab GPU

Builds and runs the CUDA solver (`src/ANPDJTLED_CUDA.cu`) on Colab's NVIDIA GPU, checks the result against the reference, and downloads the output.

**Before you start:** this notebook needs a GPU runtime. It asks for one automatically; if step 1 says there is no GPU, choose **Runtime > Change runtime type > T4 GPU** and run step 1 again.

Run the cells from top to bottom.

## 1. Check the GPU and the CUDA compiler

In [ ]:
import shutil, subprocess
if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU in this runtime. Choose Runtime > Change runtime type > T4 GPU, then run this cell again.")
if shutil.which("nvcc") is None:
    raise SystemExit("The CUDA compiler (nvcc) is missing from this runtime, so the solver cannot be built here.")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2

## 2. Get the project

Two ways; pick one:

- **From GitHub:** put the repository address in `REPO_URL` below (public repositories only).
- **Upload a zip:** leave `REPO_URL` empty. The cell asks for a file: on your Mac, right-click the project folder, choose **Compress**, and pick the `.zip` it makes.

Programs in `bin/` and files in `results/` that came from another computer are removed from the Colab copy, because programs built on a Mac cannot run here. Your own computer is not touched.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<user>/ANP-DJTLED-GPU"; leave empty to upload a zip instead

import glob, os, shutil, subprocess, zipfile
os.chdir("/content")
if REPO_URL.strip():
    name = REPO_URL.strip().rstrip("/").split("/")[-1].removesuffix(".git")
    project = os.path.join("/content", name)
    if not os.path.isdir(project):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL.strip(), project], check=True)
else:
    from google.colab import files
    print("Choose the .zip of the project folder.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file was uploaded. Run this cell again and choose the zip.")
    shutil.rmtree("/content/project", ignore_errors=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall("/content/project")
    found = [p for p in glob.glob("/content/project/**/run.py", recursive=True) if "__MACOSX" not in p]
    if not found:
        raise SystemExit("run.py is not in the zip. Zip the whole project folder (the one that contains run.py and src/).")
    project = os.path.dirname(min(found, key=len))

os.chdir(project)
missing = [p for p in ("run.py", "compare.py", "src/ANPDJTLED_CUDA.cu", "src/vtk_output.h", "models") if not os.path.exists(p)]
if missing:
    raise SystemExit(f"The project in {project} is missing: {', '.join(missing)}")
for folder in ("bin", "results"):  # built or written on another computer
    shutil.rmtree(folder, ignore_errors=True)
print("Project folder:", project)
!python3 run.py --list

## 3. Build and run on the GPU

`run.py` compiles the solver with `nvcc` the first time, then runs it. The options are the same as on your computer:

| Option | Meaning |
|---|---|
| `-a NAME ...` | which solvers to run |
| `-m MODEL ...` | which models (default: every file in `models/`) |
| `--final` | keep the final state (`..._u.vtk`, `..._undeformed.vtk`) |
| `--animation` | keep the animation (`..._simulation.vtk.series` and the timesteps folder); `--frames N` sets how many frames |
| `--times` | print the computation times and save `computation_times.csv` |
| `--repeat N` | run N times and report the median time |

In [ ]:
!python3 run.py -a ANPDJTLED_CUDA --final --animation --times

## 4. Check the result against the reference

Compares the GPU's final state with `reference results/U_ANP_DJTLED.vtk`: the maximum displacement in x, y, z and magnitude, and the largest node-by-node difference. This needs the `reference results/` folder to be part of the project you loaded in step 2.

In [ ]:
!python3 compare.py -a ANPDJTLED_CUDA

## 5. GPU against CPU on this machine

Runs every solver (the GPU one and the OpenMP CPU ones) three times and keeps their final states, then compares the two:

- **Speed:** the times table sets the GPU beside the CPU solvers on the same Colab machine. Colab's CPUs have few cores, so the CPU times here are not comparable with your own computer's.
- **Accuracy:** `compare.py --cpu-gpu` compares each GPU result with the CPU result of the same formulation and model (maximum displacement in x, y, z and magnitude, and the largest node-by-node difference).

In [ ]:
!python3 run.py --final --times --repeat 3
!python3 compare.py --cpu-gpu

## 6. Download the results

Zips the `results/` folder and downloads it. It holds the VTK files, `computation_times.csv` and the comparison tables. Open the `..._simulation.vtk.series` file in ParaView to play the animation.

In [ ]:
import os, shutil
from google.colab import files
if not os.path.isdir("results") or not os.listdir("results"):
    raise SystemExit("There is nothing in results/ yet. Run step 3 first.")
archive = shutil.make_archive("/content/results_colab", "zip", "results")
files.download(archive)